<a href="https://colab.research.google.com/github/pe8sutd/DSL26pub/blob/main/pnrg/lfsr16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Environmental Setup

In [ ]:
!pip install git+https://github.com/pe8sutd/eda4u &> /dev/null
!git clone https://github.com/pe8sutd/eda4u &> /dev/null
%load_ext plugin

In [ ]:
!pip install vcdvcd
import vcdvcd
print('vcdvcd successfully installed and imported.')

#LFSR 16

##Verilog for LFSR 16 (only a quick example, you need to create yourown PRNG)

In [50]:
%%verilog

module LFSR(input clk, rst, output reg [15:0] op);
  always@(posedge clk) begin
    if(rst) op <= 16'hf;
    else op = {op[14:0],(op[15]^op[13]^op[12]^op[10])};
  end
endmodule

/////////////////////////////////////////////////////////////////////////////////////

module TB;
  reg clk, rst;
  wire [15:0] op;

  LFSR lfsr1(clk, rst, op);

  initial begin
    clk = 0; rst = 1;
    #5 rst = 0;
    #400000; $finish;
  end

  always #1 clk=~clk;

  initial begin
    $dumpfile("dump.vcd"); $dumpvars;
  end
endmodule

VCD info: dumpfile dump.vcd opened for output.



## Read in the data from vcd file

In [ ]:
vcd_data = vcdvcd.VCDVCD('/content/dump.vcd')
print("Available signal names:")
for signal_obj in vcd_data.signals:
    print(signal_obj)

## Check the data

In [ ]:
lfsr_signal = vcd_data['TB.lfsr1.op[15:0]']
lfsr_outputs = []

# vcdvcd stores signal values as a list of (time, value) tuples in .tv property.
for time, value in lfsr_signal.tv:
    if 'x' not in value and 'z' not in value:
        try:
            lfsr_outputs.append(int(value, 2))
        except ValueError:
            # This catches cases where value might be unexpected but not 'x' or 'z'
            # For example, if it's an empty string or malformed binary, though less likely with vcdvcd
            print(f"Warning: Could not convert value '{value}' to integer at time {time}. Skipping.")

print(f"Total extracted LFSR output values: {len(lfsr_outputs)}")
print(f"First 10 LFSR output values: {lfsr_outputs[:10]}")

## Plot the histogram (this plotting takes a very long time,....)

In [ ]:
#very long simulation, to plot the histogram (donot run if no time)

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(lfsr_outputs, bins=range(min(lfsr_outputs), max(lfsr_outputs) + 2), align='left', rwidth=0.8)
plt.xlabel('LFSR Output Value')
plt.ylabel('Frequency')
plt.title('Frequency Distribution of 16-bit LFSR Outputs')
#plt.xticks(range(min(lfsr_outputs), max(lfsr_outputs) + 1))
plt.xticks([])
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
import numpy as np

unique_values_set = set(lfsr_outputs)
unique_values_str = ", ".join(map(str, sorted(list(unique_values_set))))
min_value = min(lfsr_outputs)
max_value = max(lfsr_outputs)
has_0_presence = "was not" if 0 not in lfsr_outputs else "was"
num_unique_values = len(unique_values_set)

# For verification in markdown, we can directly compare.
expected_unique_values = set(range(1, 65536)) # For a 16-bit maximal LFSR, 1 to 65535 (0,1,...,65535)
verification_match_expected = unique_values_set == expected_unique_values

print(f"Unique values: {unique_values_str}")
print(f"Min value: {min_value}")
print(f"Max value: {max_value}")
print(f"0 presence: {has_0_presence}")
print(f"Number of unique states: {num_unique_values}")
print(f"Matches expected 1-65535: {verification_match_expected}")

In [ ]:
import numpy as np

# Initialize dictionary for observed frequencies for values 1 to 65535
observed_frequencies_dict = {i: 0 for i in range(1, 65536)}

# Populate frequencies from lfsr_outputs
for output_value in lfsr_outputs:
    if output_value in observed_frequencies_dict:
        observed_frequencies_dict[output_value] += 1

# Convert dictionary values to a NumPy array in the correct order (1 to 65535)
# Ensure the order by iterating through sorted keys or by explicitly accessing keys 1 through 65535
observed_frequencies = np.array([observed_frequencies_dict[i] for i in range(1, 65536)])

print("Observed Frequencies (for values 1 to 65535):")
print(observed_frequencies)

In [ ]:
import numpy as np

total_samples = len(lfsr_outputs)
num_states = 2**16 - 1  # Correct for a maximal LFSR, excluding state 0. (2^16 - 1 = 65535)

# Calculate the expected frequency for each state
expected_freq_per_state = total_samples / num_states

# Create a NumPy array with the expected frequencies
expected_frequencies = np.full(num_states, expected_freq_per_state)

print(f"Total number of samples: {total_samples}")
print(f"Number of unique states (1-65535): {num_states}")
print(f"Expected frequency per state: {expected_freq_per_state:.2f}")
print("Expected Frequencies (for values 1 to 65535):")
print(expected_frequencies)

In [ ]:
from scipy.stats import chisquare

# Perform the Chi-Square test
chi_square_statistic, p_value = chisquare(f_obs=observed_frequencies, f_exp=expected_frequencies)

# Print the results
print(f"Chi-Square Statistic: {chi_square_statistic:.4f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
import numpy as np

# 1. Convert the lfsr_outputs list to a NumPy array
lfsr_outputs_array = np.array(lfsr_outputs)

# 2. Divide each value by 65536 to normalize them to the range [0, 1)
# Since the maximum value in lfsr_outputs is 65535, dividing by 65536 ensures the maximum normalized value is 15/16 < 1.
normalized_lfsr_outputs = lfsr_outputs_array / 65536.0

# 3. Store these values in normalized_lfsr_outputs (already done in the previous step)

# 4. Print the first 10 values and the length of normalized_lfsr_outputs
print(f"First 10 normalized LFSR output values: {normalized_lfsr_outputs[:10]}")
print(f"Total length of normalized LFSR output values: {len(normalized_lfsr_outputs)}")

## Box-Muller Transform

In [ ]:
import numpy as np

# Ensure normalized_lfsr_outputs is a NumPy array (it already is from previous step)
# If it were a list, we would convert it: normalized_lfsr_outputs = np.array(normalized_lfsr_outputs)

# Adjust the number of samples to be even for pairing
num_samples = len(normalized_lfsr_outputs)
if num_samples % 2 != 0:
    # Discard the last element to ensure an even number for pairing
    normalized_lfsr_outputs = normalized_lfsr_outputs[:-1]
    num_samples = len(normalized_lfsr_outputs)

# Split the normalized_lfsr_outputs array into two equal halves
half_point = num_samples // 2
u1 = normalized_lfsr_outputs[:half_point]
u2 = normalized_lfsr_outputs[half_point:]

# Apply the Box-Muller transform. Ensure u1 elements are not zero for log calculation.
# In a proper LFSR, 0 is typically excluded, so u1 should not contain 0.
# If u1 could contain 0, a small epsilon would be added to prevent log(0).
# Since LFSR outputs range from 1 to 15, normalized values are > 0.

# Apply Box-Muller formulas
z0 = np.sqrt(-2 * np.log(u1)) * np.cos(2 * np.pi * u2)
z1 = np.sqrt(-2 * np.log(u1)) * np.sin(2 * np.pi * u2)

# Combine z0 and z1 into a single NumPy array
guassian_numbers = np.concatenate((z0, z1))

# Print the first 10 values and the total length
print(f"First 10 Gaussian numbers: {guassian_numbers[:10]}")
print(f"Total length of Gaussian numbers: {len(guassian_numbers)}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# Set default font size for all text elements
mpl.rcParams['font.size'] = 16 # Sets a base size for all text

# Set font weight globally to 'bold'
# Note: this applies to all text, including tick labels, which might be too much.
mpl.rcParams['font.weight'] = 'bold'

# Specific adjustments for legend and axis labels
mpl.rcParams['axes.labelsize'] = 24          # Fontsize of the x and y labels
mpl.rcParams['axes.labelweight'] = 'bold'    # Weight of the axis labels
mpl.rcParams['legend.fontsize'] = 16         # Fontsize for legend items
mpl.rcParams['legend.title_fontsize'] = 16   # Fontsize for the legend title
##

plt.figure(figsize=(10, 6))
plt.hist(guassian_numbers, bins=50, density=True, alpha=0.6, color='g', label='Histogram of Samples')

# Optional: Overlay a theoretical Gaussian curve for comparison
# Calculate mean and standard deviation from the generated numbers
mu, sigma = guassian_numbers.mean(), guassian_numbers.std()

# Create a range of x values for the theoretical curve
x = np.linspace(guassian_numbers.min(), guassian_numbers.max(), 100)

# Calculate the probability density function (PDF) for the theoretical Gaussian curve
from scipy.stats import norm
plt.plot(x, norm.pdf(x, mu, sigma), 'r--', linewidth=2, label='Gaussian Fit')

plt.xlabel('Gaussian Value')
plt.ylabel('Frequency (Density)')
plt.title('Histogram of Gaussian Distributed\n Numbers from LFSR Outputs')
plt.legend()
plt.grid(False)
plt.tight_layout()
plt.show()

## Central Limit Theorem

In [ ]:
import numpy as np

# 1. Convert the lfsr_outputs list to a NumPy array if it's not already one.
#    It's currently a list, so convert it.
lfsr_outputs_array = np.array(lfsr_outputs)

# 2. Define a sample_size for the Central Limit Theorem
sample_size = 10

# 3. Calculate the number of full samples that can be drawn
num_full_samples = len(lfsr_outputs_array) // sample_size

# 4. Initialize an empty list to store the sums
clt_gaussian_numbers = []

# 5. Iterate through the lfsr_outputs array, taking sample_size consecutive elements at each step.
# 6. For each set of sample_size elements, calculate their sum.
# 7. Append each calculated sum to the clt_gaussian_numbers list.
for i in range(num_full_samples):
    start_index = i * sample_size
    end_index = start_index + sample_size
    sample = lfsr_outputs_array[start_index:end_index]
    clt_gaussian_numbers.append(np.sum(sample))

# Convert to numpy array for consistency and easier manipulation later
clt_gaussian_numbers = np.array(clt_gaussian_numbers)

# 8. Print the first 10 values and the total length of clt_gaussian_numbers
print(f"First 10 CLT Gaussian numbers: {clt_gaussian_numbers[:10]}")
print(f"Total length of CLT Gaussian numbers: {len(clt_gaussian_numbers)}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.stats import norm

# Set default font size for all text elements
mpl.rcParams['font.size'] = 16 # Sets a base size for all text

# Set font weight globally to 'bold'
# Note: this applies to all text, including tick labels, which might be too much.
mpl.rcParams['font.weight'] = 'bold'

# Specific adjustments for legend and axis labels
mpl.rcParams['axes.labelsize'] = 24          # Fontsize of the x and y labels
mpl.rcParams['axes.labelweight'] = 'bold'    # Weight of the axis labels
mpl.rcParams['legend.fontsize'] = 16         # Fontsize for legend items
mpl.rcParams['legend.title_fontsize'] = 16   # Fontsize for the legend title
##

plt.figure(figsize=(10, 6))

# 2. Create a histogram of the clt_gaussian_numbers array
plt.hist(clt_gaussian_numbers, bins=50, density=True, alpha=0.6, color='g', label='Histogram of \n Sample Sums')

# 3. Calculate the mean and standard deviation from the generated numbers
mu, sigma = clt_gaussian_numbers.mean(), clt_gaussian_numbers.std()

# 5. Create a range of x values for the theoretical curve
x = np.linspace(clt_gaussian_numbers.min(), clt_gaussian_numbers.max(), 100)

# 6. Plot a theoretical Gaussian Probability Density Function (PDF) curve
plt.plot(x, norm.pdf(x, mu, sigma), 'r--', linewidth=2, label='Gaussian Fit')

# 7. Add labels, title
plt.xlabel('Summed Value')
plt.ylabel('Frequency (Density)')
plt.title('Histogram of Sums from \n Central Limit Theorem (LFSR Outputs)')

# 8. Add a legend
plt.legend()
plt.grid(True)
plt.tight_layout()

# 9. Display the plot
plt.show()


In [ ]:
from scipy.stats import probplot
import matplotlib.pyplot as plt

# Set default font size for all text elements
mpl.rcParams['font.size'] = 16 # Sets a base size for all text

# Set font weight globally to 'bold'
# Note: this applies to all text, including tick labels, which might be too much.
mpl.rcParams['font.weight'] = 'bold'

# Specific adjustments for legend and axis labels
mpl.rcParams['axes.labelsize'] = 24          # Fontsize of the x and y labels
mpl.rcParams['axes.labelweight'] = 'bold'    # Weight of the axis labels
mpl.rcParams['legend.fontsize'] = 16         # Fontsize for legend items
mpl.rcParams['legend.title_fontsize'] = 16   # Fontsize for the legend title
##

# Create a figure and an axes object for the plot
fig, ax = plt.subplots(figsize=(10, 6))

# Generate the Q-Q plot
probplot(clt_gaussian_numbers, dist="norm", plot=ax)

# Add a title and labels
ax.set_title('Q-Q Plot of CLT Gaussian Numbers')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')

# Display the plot
plt.grid(True)
plt.tight_layout()
plt.show()

## Period

In [ ]:
import numpy as np

# Ensure lfsr_outputs is available (it should be from previous cells)
if not lfsr_outputs:
    print("LFSR outputs list is empty. Cannot check for repetitions.")
else:
    # Use a dictionary (hash table) to store the first occurrence of each state
    seen_states = {}
    potential_period = -1
    repeating_state = None
    start_index_of_repeat = -1

    for i, state in enumerate(lfsr_outputs):
        if state in seen_states:
            # A state has repeated, indicating the start of a cycle
            start_index_of_repeat = seen_states[state]
            potential_period = i - start_index_of_repeat
            repeating_state = state
            break # Found the first repeat, so the period is established
        else:
            seen_states[state] = i

    if potential_period != -1:
        print(f"State '{repeating_state}' first appeared at index {start_index_of_repeat} and repeated at index {i}.")
        print(f"This suggests a potential period of {potential_period}.")

        # Verify if the sequence actually repeats consistently for at least one full period
        is_repeating_consistently = True
        # Determine how many elements we can check for consistency
        elements_to_check = len(lfsr_outputs) - (start_index_of_repeat + potential_period)
        check_length = min(potential_period, elements_to_check)

        if check_length <= 0: # Not enough data for a second full cycle to compare after the detected repeat
             print("Not enough data to confirm full cycle repetition after the first detected repeat.")
             is_repeating_consistently = False
        else:
            for k in range(check_length):
                if lfsr_outputs[start_index_of_repeat + k] != lfsr_outputs[start_index_of_repeat + potential_period + k]:
                    is_repeating_consistently = False
                    break

        if is_repeating_consistently:
            print(f"The sequence is confirmed to be repeating with a period of {potential_period}.")
            print(f"Total length of output: {len(lfsr_outputs)}")

            # Calculate number of full periods from the start of the repeating sequence
            num_elements_in_repeating_part = len(lfsr_outputs) - start_index_of_repeat
            num_full_periods_observed = num_elements_in_repeating_part // potential_period
            remaining_elements_after_full_periods = num_elements_in_repeating_part % potential_period

            print(f"Number of full periods observed from the repeating state: {num_full_periods_observed}")
            print(f"Remaining elements after full periods: {remaining_elements_after_full_periods}")
        else:
            print("The sequence does NOT consistently repeat with the detected potential period.")
            print("This might indicate that the detected 'repeat' was a false positive,")
            print("or the period calculation is incorrect, or the LFSR is not maximal.")
    else:
        print("No repetition of any state found within the sequence.")
        print("This is unexpected for an LFSR run longer than its maximal period.")

## Summary


- LFSR Outputs Extraction: We extracted 200,002 16-bit output values from the Verilog simulation.     
- Uniqueness and Range: The LFSR generated 65,535 unique values ranging from 1 to 65,535, as expected for a maximal length LFSR of 16 bits (2^16 - 1 states), excluding the all-zero state.     
- Chi-Square Test: A Chi-Square test was performed to assess the uniformity of the LFSR outputs. The results (Chi-Square Statistic: 1055.4034, P-value: 1.0000) indicate that the distribution of generated numbers is consistent with a uniform distribution, suggesting good randomness properties.     
- Box-Muller Transform: We applied the Box-Muller transform to the normalized LFSR outputs to generate Gaussian-distributed numbers. The histogram of these transformed numbers showed a clear bell-shaped curve, confirming the transformation's success.      
- Central Limit Theorem (CLT): By summing groups of 10 consecutive LFSR outputs, we demonstrated the Central Limit Theorem. The histogram of these sums also approximated a Gaussian distribution, further supported by a Q-Q plot which showed linearity, indicating the sums are normally distributed.      
- Period Detection: Using hash table tracking, we confirmed that the LFSR sequence has a period of 65,535, which is the maximal length for a 16-bit LFSR. We observed 3 full periods and 3397 remaining elements within the simulated output length.      


# End, March 8th 2026